In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

import os
import getpass

if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass("Enter Groq API Key: ")
if not os.environ.get('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = getpass.getpass("Enter Google API Key: ")
if not os.environ.get('TAVILY_API_KEY'):
    os.environ['TAVILY_API_KEY'] = getpass.getpass("Enter Tavily API Key: ")

Enter Groq API Key:  ········
Enter Google API Key:  ········
Enter Tavily API Key:  ········


In [2]:
%pip install -q langgraph langchain-tavily

Note: you may need to restart the kernel to use updated packages.


In [5]:
# Loading the Vectorstore — CRAG's base retrieval is dense (Chroma)
# here; the correction layer itself is retriever-agnostic, so
# colbert_vectorstore from 08/09 could be substituted in just as
# easily (anything exposing .as_retriever()/.invoke() works).
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorstore = Chroma(
    embedding_function=embeddings,
    #persist_directory=str(project_root / "data" / "chroma_naive_gemini"),
    persist_directory= "~/Codes/rag-lab/data/chroma_naive_gemini",
)
print("Chunks in store:", vectorstore._collection.count())

llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0)

Chunks in store: 576


In [6]:
# Retrieval Grader — binary relevance check via structured output.
# Uses ChatGroq specifically for grading/rewriting (NOT the `llm`
# above): structured output needs reliable function-calling support.
# gemini-3.6-flash's fixed-sampling behavior (see the UserWarning you
# already hit in 01_foundation.ipynb) and generation.ipynb's
# "orinth-1.5:9B" (unknown tool-calling support) are both less certain
# bets here — Groq's llama-3.3 is already proven working for
# structured output in this project (08_colbert.ipynb's smoke test).
from typing import List, Optional, TypedDict
from pydantic import BaseModel, Field
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

#grader_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

grader_llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

class GradeDocuments(BaseModel):
    """Binary score for relevance check on a retrieved document."""
    binary_score: str = Field(
        description="'yes' if the document is relevant to the question, else 'no'"
    )


structured_grader = grader_llm.with_structured_output(GradeDocuments)

grade_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a grader assessing relevance of a retrieved document to a "
     "user question. If the document contains keywords or semantic "
     "meaning related to the question, grade it as relevant. Give a "
     "binary score 'yes' or 'no'."),
    ("human", "Retrieved document:\n\n{document}\n\nUser question: {question}"),
])
retrieval_grader = grade_prompt | structured_grader

In [7]:
# Sanity check the grader against a real retrieved chunk before
# wiring it into the graph
test_query = "Why do PINNs struggle with irregular geometry?"
_docs = vectorstore.as_retriever(search_kwargs={"k": 4}).invoke(test_query)
print(retrieval_grader.invoke({"question": test_query, "document": _docs[0].page_content}))

binary_score='no'


In [8]:
# Question Re-writer — used only when at least one retrieved doc is
# graded irrelevant, to optimize the query for web search specifically
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a question re-writer that converts an input question into "
     "a better version optimized for web search. Reason about the "
     "underlying semantic intent before rewriting."),
    ("human", "Here is the initial question:\n\n{question}\n\nFormulate an improved question."),
])
question_rewriter = rewrite_prompt | grader_llm | StrOutputParser()

print(question_rewriter.invoke({"question": test_query}))

What challenges cause physics‑informed neural networks (PINNs) to perform poorly on problems with irregular or complex geometries?


In [9]:
# Web Search — langchain-tavily (current package; the old
# langchain_community.tools.tavily_search.TavilySearchResults import
# is deprecated and returns a different shape, a plain list rather
# than a {"results": [...]} dict — the web_search node below is
# written for THIS package's return shape).
from langchain_tavily import TavilySearch

web_search_tool = TavilySearch(max_results=3)

In [10]:
class CRAGState(TypedDict):
    """
    question: the current question (gets rewritten if web search triggers)
    generation: the LLM's final answer
    web_search: "Yes"/"No" — set by grade_documents, read by the conditional edge
    documents: current working set of retrieved (and possibly web-augmented) docs
    """
    question: str
    generation: str
    web_search: str
    documents: List[Document]

In [11]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

answer_prompt = ChatPromptTemplate.from_template(
    "Answer the question based only on the following context:\n"
    "{context}\n\nQuestion: {question}"
)


def retrieve(state: CRAGState) -> dict:
    print("---RETRIEVE---")
    question = state["question"]
    documents = retriever.invoke(question)
    return {"documents": documents, "question": question}


def grade_documents(state: CRAGState) -> dict:
    print("---CHECK DOCUMENT RELEVANCE TO QUESTION---")
    question = state["question"]
    documents = state["documents"]
    filtered_docs = []
    web_search = "No"
    for d in documents:
        score = retrieval_grader.invoke({"question": question, "document": d.page_content})
        if score.binary_score.strip().lower() == "yes":
            print("---GRADE: DOCUMENT RELEVANT---")
            filtered_docs.append(d)
        else:
            print("---GRADE: DOCUMENT NOT RELEVANT---")
            web_search = "Yes"
    return {"documents": filtered_docs, "question": question, "web_search": web_search}


def transform_query(state: CRAGState) -> dict:
    print("---TRANSFORM QUERY---")
    question = state["question"]
    documents = state["documents"]
    better_question = question_rewriter.invoke({"question": question})
    return {"documents": documents, "question": better_question}


def web_search(state: CRAGState) -> dict:
    print("---WEB SEARCH---")
    question = state["question"]
    documents = state["documents"]
    tavily_results = web_search_tool.invoke({"query": question})
    web_content = "\n".join(r["content"] for r in tavily_results.get("results", []))
    documents = documents + [Document(page_content=web_content, metadata={"source": "web_search"})]
    return {"documents": documents, "question": question}


def generate(state: CRAGState) -> dict:
    print("---GENERATE---")
    question = state["question"]
    documents = state["documents"]
    context = "\n\n".join(d.page_content for d in documents)
    chain = answer_prompt | llm | StrOutputParser()
    generation = chain.invoke({"context": context, "question": question})
    return {"documents": documents, "question": question, "generation": generation}


def decide_to_generate(state: CRAGState) -> str:
    print("---ASSESS GRADED DOCUMENTS---")
    if state["web_search"] == "Yes":
        print("---DECISION: NOT ALL DOCUMENTS RELEVANT, TRANSFORM QUERY---")
        return "transform_query"
    print("---DECISION: GENERATE---")
    return "generate"

In [12]:
from langgraph.graph import END, START, StateGraph

workflow = StateGraph(CRAGState)
workflow.add_node("retrieve", retrieve)
workflow.add_node("grade_documents", grade_documents)
workflow.add_node("generate", generate)
workflow.add_node("transform_query", transform_query)
workflow.add_node("web_search_node", web_search)

workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "grade_documents")
workflow.add_conditional_edges(
    "grade_documents",
    decide_to_generate,
    {"transform_query": "transform_query", "generate": "generate"},
)
workflow.add_edge("transform_query", "web_search_node")
workflow.add_edge("web_search_node", "generate")
workflow.add_edge("generate", END)

app = workflow.compile()

In [13]:
# Test 1: a question your corpus should answer directly — expect
# every retrieved doc graded relevant, web search skipped entirely.
inputs = {"question": "Why do PINNs struggle with irregular geometry?"}
for output in app.stream(inputs):
    for key in output.keys():
        print(f"Node '{key}' finished")
print("\n--- FINAL ANSWER ---")
print(output[key]["generation"])

---RETRIEVE---
Node 'retrieve' finished
---CHECK DOCUMENT RELEVANCE TO QUESTION---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---ASSESS GRADED DOCUMENTS---
---DECISION: NOT ALL DOCUMENTS RELEVANT, TRANSFORM QUERY---
Node 'grade_documents' finished
---TRANSFORM QUERY---
Node 'transform_query' finished
---WEB SEARCH---


/home/mohnish/my-jupyter-env/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Node 'web_search_node' finished
---GENERATE---
Node 'generate' finished

--- FINAL ANSWER ---
Based on the provided context, PINNs face performance issues and challenges with complex or irregular geometries due to the following factors:

* **Lack of topological awareness:** There is no clear way to inform standard PINNs about the topology of the domain where the problem is being solved.
* **Sensitivity to domain changes:** PINNs can be extremely sensitive to changes in the computational domain, causing them to output incorrect results.
* **Boundary condition and geometry incorporation:** Practical implementations struggle with incorporating complex geometries and enforcing boundary conditions.
* **Inconsistent loss functions:** Challenges arise when formulating consistent loss functions due to the heterogeneity of loss function components (which account for PDE residuals, boundaries, etc.).
* **Multiphysics phenomena:** In real-world problems, PINNs face issues related to complex domai

In [14]:
# Test 2: something clearly outside your PINN/GNN paper corpus —
# expect at least one doc graded irrelevant, triggering
# transform_query -> web_search_node before generation.
inputs = {"question": "What is the current version of PyTorch?"}
for output in app.stream(inputs):
    for key in output.keys():
        print(f"Node '{key}' finished")
print("\n--- FINAL ANSWER ---")
print(output[key]["generation"])

---RETRIEVE---
Node 'retrieve' finished
---CHECK DOCUMENT RELEVANCE TO QUESTION---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---ASSESS GRADED DOCUMENTS---
---DECISION: NOT ALL DOCUMENTS RELEVANT, TRANSFORM QUERY---
Node 'grade_documents' finished
---TRANSFORM QUERY---
Node 'transform_query' finished
---WEB SEARCH---
Node 'web_search_node' finished
---GENERATE---


/home/mohnish/my-jupyter-env/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Node 'generate' finished

--- FINAL ANSWER ---
Based on the provided context, as of 2025, the current stable version of PyTorch is **2.8.0** (released August 6, 2025). 

*(Note: The provided text also mentions version 1.11.0 and lists "Stable (2.7.0)" in a package selector section).*


In [15]:
# The nodes/graph above are the readable, step-by-step version.
# rag_lab/strategies/crag.py packages this as CRAGStrategy, exposing
# the same .retrieve()/.run() interface as every other strategy —
# save that file, then CRAG becomes pluggable into generation.ipynb's
# strategies dict exactly like ColBERTStrategy in 09_colbert_eval.ipynb,
# for a full 8-way comparison in scoring.ipynb.
from rag_lab.strategies.crag import CRAGStrategy

crag_strategy = CRAGStrategy(vectorstore, llm=llm, grader_llm=grader_llm)
print(crag_strategy.run("Why do PINNs struggle with irregular geometry?"))

/home/mohnish/my-jupyter-env/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Based on the provided context, the main difficulties and issues PINNs face when handling complex or irregular geometries include:

1. **Extreme Sensitivity to Domain Changes:** PINNs can be extremely sensitive to changes in the computational domain, which can cause them to output completely wrong results.
2. **High Computational Expense:** Complex geometries with an excessive number of facets require mesh simplification to manage the substantial computational cost of calculating the Signed Distance Function (SDF).
3. **Multiphysics and Geometry Issues:** Real-world applications of PINNs inherently face challenges related to the complex geometry of computational domains combined with multiphysics phenomena.
4. **Impact on Accuracy and Constraints:** Handling geometric information properly is complex but crucial, as it directly affects training accuracy and the proper incorporation of physical laws (PDEs) and boundary conditions (BCs). 

*(Note: The text also emphasizes that PINNs are of